# Classification V1 Baseline
## 향후 3개월 지역 주택가격 방향성 분류

---

## 1. 목적

본 분류 모델의 목적은 개별 아파트의 미래 가격을 직접 단정적으로 예측하는 것이 아니라,

> **현재 시점의 시장·주택·입지 정보를 바탕으로 해당 지역의 향후 3개월 가격 흐름을 확률적으로 판단하는 것**

이다.

본 프로젝트의 세 모델은 서로 다른 역할을 담당한다.

- **Regression**
  - 개별 아파트의 예상 거래가격 예측

- **Clustering**
  - 아파트 단지의 주택·입지 특성을 기반으로 시장 유형 분류

- **Classification**
  - 해당 지역의 향후 3개월 가격 방향성 판단

최종적으로 다음과 같은 형태의 서비스를 목표로 한다.

```text
아파트 조건 입력
        ↓
[Clustering]
단지 유형
        ↓
[Regression]
예상 거래가격
        ↓
[Classification]
향후 3개월 지역 시장 방향성
        ↓
가격 + 단지 유형 + 시장 흐름 제공
````

예시:

```text
예상 거래가격
→ 14.2억원

단지 유형
→ 업무지·역세권 고가형

향후 3개월 지역 시장
→ 상승 가능성 68%

시장 판단
→ 상승 우세
```

---

# 2. Classification Target 설계

분류 Target은 **현재 시점과 향후 3개월의 지역 대표 가격 수준을 비교하여 생성**한다.

개념적으로,

$$
R_{t,3m}
=
\frac{
P_{future,3m} - P_t
}{
P_t
}
$$

* $P_t$

  * 현재 시점에서 확인 가능한 지역 가격 기준

* $P_{future,3m}$

  * 향후 3개월 동안 관측된 지역 가격 수준

* $R_{t,3m}$

  * 향후 3개월 지역 가격 변화율

이를 기반으로 상승·보합·하락 등의 시장 방향을 정의한다.

---

## 2-1. Target 세부 정의는 EDA 후 확정

현재 단계에서는 다음 요소를 임의로 고정하지 않는다.

### 지역 단위

후보:

```text
서울시 구
법정동
```

너무 세밀한 지역 단위를 사용하면 거래량 부족으로 가격지표가 불안정해질 수 있으므로,
지역별 월 거래건수와 가격 안정성을 비교한 후 결정한다.

### 대표 가격 지표

후보:

```text
중위 거래가격
중위 평당가
월별 대표 평당가
```

아파트 면적 차이의 영향을 줄이기 위해 **평당가 기반 지표를 우선 검토**하되,
실제 데이터 분포와 안정성을 확인한 뒤 확정한다.

### Class 정의

후보 1 — Binary Classification

```text
0 = 비상승
1 = 상승
```

후보 2 — Multi-class Classification

```text
0 = 하락
1 = 보합
2 = 상승
```

보합 범위 역시 임의의 ±3%와 같은 값을 바로 적용하지 않고,
실제 향후 3개월 변화율 분포를 확인한 후 결정한다.

---

# 3. 왜 개별 아파트가 아닌 지역 방향성을 예측하는가?

아파트는 동일 단지에서도 다음 조건에 따라 거래가격 차이가 크게 발생할 수 있다.

```text
전용면적
층
동
향
내부 상태
특수 거래
거래 시점
```

또한 개별 단지는 거래가 매달 발생하지 않을 수 있다.

따라서 개별 아파트의

> "3개월 뒤 가격이 오른다."

를 직접 예측하기보다,

> "해당 지역의 향후 3개월 시장 흐름이 상승 우세인지 판단한다."

는 문제로 정의한다.

이를 통해 거래량 부족으로 인한 변동성을 줄이고
모델의 비즈니스적 해석 가능성을 높이는 것을 목표로 한다.

---

# 4. 예측 결과 표현 원칙

분류 모델의 결과를 투자 판단처럼 단정적으로 표현하지 않는다.

잘못된 예:

```text
3개월 후 가격이 상승합니다.
```

사용할 표현:

```text
향후 3개월 상승 가능성: 72%
→ 상승 우세
```

또한 모델의 확률이 애매한 경우 강제로 상승 또는 하락으로 결정하지 않는
**판단 보류 영역**을 검토한다.

예:

```text
높은 상승 확률
→ 상승 우세

중간 확률
→ 방향성 불확실

낮은 상승 확률
→ 하락/비상승 우세
```

정확한 확률 Threshold는 Validation 결과를 기반으로 결정한다.

---

# 5. Data Leakage 방지 원칙

본 분류 모델은 미래 가격을 Target으로 사용하기 때문에
**시간 누수(Time Leakage) 방지가 가장 중요하다.**

## 핵심 원칙

> 시점 $t$의 예측에 사용하는 Feature는 반드시 $t$ 시점 또는 그 이전에 알 수 있는 정보만 사용한다.

향후 3개월의 거래가격은 **Target 생성에만 사용**하며 Feature에는 절대 포함하지 않는다.

---

## 5-1. 사용 금지 Feature

예측 시점 이후의 정보를 포함하는 Feature는 사용하지 않는다.

예:

```text
향후 거래가격
향후 평당가
향후 거래량
향후 전세가격
향후 금리
향후 정책 상태

Target 생성에 사용된 미래 가격 통계
향후 3개월 가격 변화율
최종 Classification Target
```

현재 거래가격에서 직접 파생되어 예측 목적과 충돌하는 변수 역시
Feature 사용 여부를 별도로 검토한다.

---

## 5-2. Lag Feature 사용 원칙

다음과 같은 과거 정보는 사용할 수 있다.

```text
전월 지역 가격
과거 3개월 지역 가격
과거 6개월 지역 가격

전세 Lag Feature
금리 Lag Feature
정책 상태 Feature

주택 특성
입지 Feature
```

Rolling 통계를 생성할 때도 현재 또는 미래 값이 섞이지 않도록

```text
shift
→ rolling
```

순서를 유지한다.

---

# 6. 시간순 데이터 분할

부동산 시장은 시간에 따라 가격 수준과 시장 환경이 변하므로
Random Train/Test Split을 사용하지 않는다.

모델 학습 및 평가는 반드시 시간 순서를 유지한다.

기본 구조:

```text
과거
│
├── Train
│
├── Validation
│
└── Test
                 미래
```

Validation을 통해

```text
Feature 구성
Model 설정
Class Threshold
Probability Threshold
```

등을 결정한 후,

**Test 데이터는 최종 성능 평가에 한 번만 사용한다.**

---

# 7. 향후 3개월 Target의 데이터 제한

향후 3개월 가격을 Target으로 사용하기 때문에
데이터 마지막 시점에서는 Target을 만들 수 없다.

예를 들어 데이터가 2026-06까지 존재한다면,

```text
2026-04
→ 2026-05~07 필요

2026-05
→ 2026-06~08 필요

2026-06
→ 2026-07~09 필요
```

와 같이 미래 데이터가 부족해진다.

따라서 **Target을 실제로 생성할 수 있는 마지막 기준월을 먼저 계산한 뒤**
Train / Validation / Test 기간을 다시 확정한다.

기존 Regression의 Test 기간을 그대로 복사하지 않고,
Classification Target의 예측 Horizon에 맞게 별도로 설계한다.

---

# 8. Classification V1 목표

V1에서는 최고 성능을 만드는 것이 목적이 아니다.

다음 사항을 우선 검증한다.

1. 지역 단위별 Target 안정성
2. 향후 3개월 변화율 분포
3. Class 비율
4. 시간 누수 여부
5. Baseline Classification 성능
6. 기간별 성능 안정성

V1 결과를 기준으로 이후 고도화를 진행한다.

---

# 9. 평가 지표

Class 구조에 따라 평가 지표를 결정한다.

Binary Classification의 경우:

```text
Accuracy
Precision
Recall
F1-score
ROC-AUC
PR-AUC
Confusion Matrix
```

Multi-class Classification의 경우:

```text
Accuracy
Macro F1
Weighted F1
Class별 Precision / Recall
Confusion Matrix
```

단순 전체 Accuracy뿐만 아니라
**월별·기간별 성능 안정성**도 함께 확인한다.

---

# 10. 추가 검증

모델이 실제 서비스에서 사용할 수 있는지 확인하기 위해
확률값 자체도 분석한다.

특히 다음을 검토한다.

```text
예측 확률 Calibration

높은 Confidence 구간의 실제 정확도

판단 보류 구간 도입 시
Coverage와 Accuracy의 Trade-off

시장 상승기 / 하락기별 성능

지역별 성능 차이
```

예를 들어 전체 데이터를 무조건 판단하는 것보다,

```text
Confidence가 충분한 60%의 데이터만 판단
→ Accuracy 상승
```

하는 구조가 실제 서비스에서는 더 적절할 수 있다.

---

# 11. V1 이후 고도화 방향

Classification V1 이후 다음을 비교한다.

```text
구 vs 법정동 Target

Binary vs 3-Class

1개월 / 3개월 / 6개월 Horizon

가격 변화율 Threshold

Feature 추가 및 제거

Clustering 결과 Feature 활용

Hyperparameter Tuning

Probability Calibration

Confidence 기반 판단 보류

SHAP 기반 방향성 요인 분석
```

---

# Classification V1 핵심 원칙

> **개별 아파트의 미래 가격을 단정적으로 예측하지 않는다.**

> **지역 시장의 향후 3개월 가격 방향을 확률적으로 판단한다.**

> **미래 가격은 Target 생성에만 사용하며, 모든 Feature는 예측 시점 이전에 확인 가능한 정보만 사용한다.**

> **Target의 지역 단위와 상승·보합·하락 기준은 임의로 정하지 않고 데이터 분포와 안정성을 확인한 뒤 결정한다.**

> **낮은 확신도의 예측은 강제로 판단하지 않는 서비스 구조를 검토한다.**



1. 데이터 로드
2. 지역 단위 후보 확인
3. 구 vs 법정동 월별 거래량 비교
4. 가격지표 후보 비교
5. 향후 3개월 변화율 생성
6. Target 분포 확인
7. Binary / 3-Class 결정
8. 시간 분할
9. Feature 누수 점검
10. Classification Baseline

In [1]:
# ============================================================
# Classification V1 - 기본 설정
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False


PROJECT_DIR = Path.cwd().parents[3]

DATA_PATH = (
    PROJECT_DIR
    / "data"
    / "v2_5year_seoul"
    / "processed"
    / "df_final_v1.parquet"
)


df = pd.read_parquet(DATA_PATH)

df["거래월일자"] = pd.to_datetime(
    df["거래월일자"]
)

print("데이터:", df.shape)

print(
    "기간:",
    df["거래월일자"].min(),
    "~",
    df["거래월일자"].max(),
)

데이터: (234302, 124)
기간: 2021-07-01 00:00:00 ~ 2026-06-01 00:00:00


In [2]:
# ============================================================
# 지역 관련 컬럼 확인
# ============================================================

location_keywords = [
    "구",
    "법정동",
    "시군구",
]

location_cols = [
    col
    for col in df.columns
    if any(
        keyword in col
        for keyword in location_keywords
    )
]

print("===== 지역 관련 컬럼 =====")

for col in location_cols:
    print("-", col)

===== 지역 관련 컬럼 =====
- 법정동시군구코드
- 법정동읍면동코드
- 법정동지번코드
- 법정동본번코드
- 법정동부번코드
- 도로명시군구코드
- 법정동
- 층_구분
- 시군구코드
- 시군구명
- 연식_구분
- 구_전월_중위가격
- 구_전월_중위평당가
- 구_전월_거래량
- 구_과거3개월_중위가격
- 구_과거6개월_중위가격
- 구_과거3개월_거래량
- 구_과거6개월_거래량
- 구_전월_가격변화율
- 역세권_구간


In [3]:
candidate_cols = [
    col
    for col in [
        "법정동시군구코드",
        "법정동",
        "구",
        "시군구",
        "평당가",
        "거래금액",
        "전용면적",
    ]
    if col in df.columns
]

display(
    df[candidate_cols]
    .head()
)

,법정동시군구코드,법정동,평당가,거래금액,전용면적
0,11110,무악동,4027.954743,140000,114.900
1,11110,사직동,4827.006666,138000,94.510
2,11110,홍파동,7384.075921,189000,84.614
3,11110,내수동,4503.515417,205000,150.480
4,11110,창신동,4218.370018,69800,54.700


In [4]:
# ============================================================
# 월 기준 생성
# ============================================================

df["기준월"] = (
    df["거래월일자"]
    .dt.to_period("M")
)

print(
    "기준월:",
    df["기준월"].min(),
    "~",
    df["기준월"].max(),
)

print(
    "전체 월 수:",
    df["기준월"].nunique(),
)

기준월: 2021-07 ~ 2026-06
전체 월 수: 60


In [5]:
# ============================================================
# 서울 25개 구 매핑
# ============================================================

gu_map = {
    11110: "종로구",
    11140: "중구",
    11170: "용산구",
    11200: "성동구",
    11215: "광진구",
    11230: "동대문구",
    11260: "중랑구",
    11290: "성북구",
    11305: "강북구",
    11320: "도봉구",
    11350: "노원구",
    11380: "은평구",
    11410: "서대문구",
    11440: "마포구",
    11470: "양천구",
    11500: "강서구",
    11530: "구로구",
    11545: "금천구",
    11560: "영등포구",
    11590: "동작구",
    11620: "관악구",
    11650: "서초구",
    11680: "강남구",
    11710: "송파구",
    11740: "강동구",
}


# 코드 자료형 통일
df["시군구코드"] = pd.to_numeric(
    df["법정동시군구코드"],
    errors="coerce",
).astype("Int64")


# 구 이름 매핑
df["구"] = (
    df["시군구코드"]
    .map(gu_map)
)


print(
    "원본 dtype:",
    df["법정동시군구코드"].dtype,
)

print(
    "변환 dtype:",
    df["시군구코드"].dtype,
)

print(
    "구 개수:",
    df["구"].nunique(),
)

print(
    "구 결측:",
    df["구"].isna().sum(),
)

print(
    sorted(
        df["구"]
        .dropna()
        .unique()
    )
)

원본 dtype: object
변환 dtype: Int64
구 개수: 25
구 결측: 0
['강남구', '강동구', '강북구', '강서구', '관악구', '광진구', '구로구', '금천구', '노원구', '도봉구', '동대문구', '동작구', '마포구', '서대문구', '서초구', '성동구', '성북구', '송파구', '양천구', '영등포구', '용산구', '은평구', '종로구', '중구', '중랑구']


In [6]:
# ============================================================
# 구 × 월 시장 가격 테이블
# ============================================================

gu_month_df = (
    df
    .groupby(
        ["구", "기준월"],
        observed=True,
    )
    .agg(
        거래건수=(
            "transaction_id",
            "size",
        ),
        월중위평당가=(
            "평당가",
            "median",
        ),
    )
    .reset_index()
)


print(
    "구 × 월 관측치:",
    f"{len(gu_month_df):,}",
)

print(
    "구 개수:",
    gu_month_df["구"].nunique(),
)

print(
    "월 개수:",
    gu_month_df["기준월"].nunique(),
)

display(
    gu_month_df.head()
)

구 × 월 관측치: 1,500
구 개수: 25
월 개수: 60


,구,기준월,거래건수,월중위평당가
0,강남구,2021-07,205,9022.799071
1,강남구,2021-08,194,7774.187911
2,강남구,2021-09,142,8460.171952
3,강남구,2021-10,124,9497.832595
4,강남구,2021-11,106,7257.751704


In [7]:
# ============================================================
# 구별 월 거래건수 안정성
# ============================================================

gu_volume_profile = (
    gu_month_df
    .groupby("구")["거래건수"]
    .agg(
        월수="count",
        최소="min",
        중앙값="median",
        평균="mean",
        최대="max",
    )
    .sort_values("중앙값")
)


display(
    gu_volume_profile.round(1)
)

,월수,최소,중앙값,평균,최대
구,,,,,
종로구,60,4,31.0,34.9,94
중구,60,5,52.0,58.4,186
금천구,60,9,54.5,61.0,165
용산구,60,8,59.5,69.8,272
강북구,60,8,64.5,73.0,202
광진구,60,7,65.5,86.3,324
중랑구,60,16,96.5,117.5,383
관악구,60,14,99.0,118.2,369
도봉구,60,16,99.5,116.6,391


In [8]:
# ============================================================
# 전체 구 × 월 거래량 분포
# ============================================================

display(
    gu_month_df["거래건수"]
    .describe()
    .to_frame()
    .round(1)
)

,거래건수
count,1500.0
mean,156.2
std,143.4
min,4.0
25%,53.0
50%,117.0
75%,205.0
max,1018.0


In [9]:
# ============================================================
# 최근 3개월 / 향후 3개월 시장 가격 수준
# ============================================================

gu_month_df = (
    gu_month_df
    .sort_values(
        ["구", "기준월"]
    )
    .reset_index(drop=True)
)


# 현재 포함 최근 3개월 월 중위 평당가
gu_month_df["최근3개월_평당가"] = (
    gu_month_df
    .groupby("구")["월중위평당가"]
    .transform(
        lambda x: (
            x.rolling(
                window=3,
                min_periods=3,
            )
            .median()
        )
    )
)


# 향후 1~3개월 월 중위 평당가
gu_month_df["미래1개월_평당가"] = (
    gu_month_df
    .groupby("구")["월중위평당가"]
    .shift(-1)
)

gu_month_df["미래2개월_평당가"] = (
    gu_month_df
    .groupby("구")["월중위평당가"]
    .shift(-2)
)

gu_month_df["미래3개월_평당가"] = (
    gu_month_df
    .groupby("구")["월중위평당가"]
    .shift(-3)
)


# 미래 3개월 대표 가격
gu_month_df["향후3개월_평당가"] = (
    gu_month_df[
        [
            "미래1개월_평당가",
            "미래2개월_평당가",
            "미래3개월_평당가",
        ]
    ]
    .median(axis=1)
)

In [10]:
# ============================================================
# 향후 3개월 Target 생성 가능 여부
# ============================================================

future_complete_mask = (
    gu_month_df[
        [
            "미래1개월_평당가",
            "미래2개월_평당가",
            "미래3개월_평당가",
        ]
    ]
    .notna()
    .all(axis=1)
)

target_available_mask = (
    gu_month_df["최근3개월_평당가"].notna()
    & future_complete_mask
)


gu_month_df["향후3개월_변화율"] = np.nan

gu_month_df.loc[
    target_available_mask,
    "향후3개월_변화율",
] = (
    (
        gu_month_df.loc[
            target_available_mask,
            "향후3개월_평당가",
        ]
        - gu_month_df.loc[
            target_available_mask,
            "최근3개월_평당가",
        ]
    )
    / gu_month_df.loc[
        target_available_mask,
        "최근3개월_평당가",
    ]
)


print(
    "전체 구×월:",
    f"{len(gu_month_df):,}",
)

print(
    "Target 생성 가능:",
    f"{target_available_mask.sum():,}",
)

print(
    "Target 생성 불가:",
    f"{(~target_available_mask).sum():,}",
)

print(
    "사용 가능 기간:",
    gu_month_df.loc[
        target_available_mask,
        "기준월",
    ].min(),
    "~",
    gu_month_df.loc[
        target_available_mask,
        "기준월",
    ].max(),
)

전체 구×월: 1,500
Target 생성 가능: 1,375
Target 생성 불가: 125
사용 가능 기간: 2021-09 ~ 2026-03


In [11]:
# ============================================================
# 향후 3개월 가격 변화율 분포
# ============================================================

target_market_df = (
    gu_month_df.loc[
        target_available_mask
    ]
    .copy()
)


display(
    (
        target_market_df[
            "향후3개월_변화율"
        ]
        * 100
    )
    .describe(
        percentiles=[
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
        ]
    )
    .to_frame("변화율_%")
    .round(2)
)

,변화율_%
count,1375.00
mean,0.73
std,8.66
min,-40.62
5%,-13.03
10%,-9.19
25%,-3.63
50%,1.03
75%,5.43
90%,9.60


In [12]:
# ============================================================
# 단순 상승 / 하락 비율 확인
# ============================================================

direction_check = pd.Series({
    "하락": (
        target_market_df[
            "향후3개월_변화율"
        ] < 0
    ).mean() * 100,

    "상승": (
        target_market_df[
            "향후3개월_변화율"
        ] > 0
    ).mean() * 100,

    "변화없음": (
        target_market_df[
            "향후3개월_변화율"
        ] == 0
    ).mean() * 100,
})

display(
    direction_check
    .to_frame("비율_%")
    .round(2)
)

,비율_%
하락,44.07
상승,55.93
변화없음,0.00


In [13]:
# ============================================================
# 보합 Threshold 후보 비교
# ============================================================

threshold_results = []

for threshold in [0.01, 0.02, 0.03]:
    change_rate = (
        target_market_df[
            "향후3개월_변화율"
        ]
    )

    decline = (
        change_rate < -threshold
    ).mean() * 100

    flat = (
        change_rate.abs() <= threshold
    ).mean() * 100

    rise = (
        change_rate > threshold
    ).mean() * 100

    threshold_results.append({
        "보합기준": f"±{threshold * 100:.0f}%",
        "하락_%": decline,
        "보합_%": flat,
        "상승_%": rise,
    })


threshold_compare = pd.DataFrame(
    threshold_results
)

display(
    threshold_compare.round(2)
)

,보합기준,하락_%,보합_%,상승_%
0,±1%,39.42,10.33,50.25
1,±2%,33.24,24.00,42.76
2,±3%,28.29,34.40,37.31


In [14]:
# ============================================================
# 변화율 주요 분위수
# ============================================================

display(
    (
        target_market_df[
            "향후3개월_변화율"
        ]
        * 100
    )
    .quantile([
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
    ])
    .to_frame("변화율_%")
    .round(2)
)

,변화율_%
0.05,-13.03
0.10,-9.19
0.25,-3.63
0.50,1.03
0.75,5.43
0.90,9.60
0.95,13.88


## 초기 3-Class Target 실험

향후 3개월 지역 가격 변화율을 기준으로 먼저
`하락 / 보합 / 상승`의 3-Class Classification 문제를 검토한다.

### 실험 기준

$$
R_{t,3m}
=
\frac{
P_{future,3m} - P_{recent,3m}
}{
P_{recent,3m}
}
$$

분류 기준은 다음과 같다.

- **하락 (0)**: 변화율 < -3%
- **보합 (1)**: -3% ≤ 변화율 ≤ +3%
- **상승 (2)**: 변화율 > +3%

실제 데이터에서 클래스 비율은 다음과 같다.

- 하락: 약 28.3%
- 보합: 약 34.4%
- 상승: 약 37.3%

세 클래스의 분포가 비교적 균형적이고 작은 가격 변동을
상승 또는 하락으로 강제하지 않는다는 점을 고려해
초기 Baseline Threshold로 ±3%를 사용한다.

이 실험 결과를 확인한 뒤, 최종 Classification V1에서는
비즈니스 목적을 더 명확히 하기 위해
**향후 3개월 +3% 초과 상승 여부를 예측하는 Binary Classification**으로 확정한다.

In [15]:
# ============================================================
# Classification V1 Target 생성
# ============================================================

CLASS_TARGET = "시장방향"
FLAT_THRESHOLD = 0.03


def make_market_direction(change_rate):
    if change_rate < -FLAT_THRESHOLD:
        return 0

    if change_rate > FLAT_THRESHOLD:
        return 2

    return 1


target_market_df[CLASS_TARGET] = (
    target_market_df[
        "향후3개월_변화율"
    ]
    .apply(make_market_direction)
    .astype("int8")
)


direction_name_map = {
    0: "하락",
    1: "보합",
    2: "상승",
}

In [16]:
# ============================================================
# Target 분포 확인
# ============================================================

target_distribution = (
    target_market_df[CLASS_TARGET]
    .value_counts()
    .sort_index()
    .rename(index=direction_name_map)
    .to_frame("구×월")
)

target_distribution["비율_%"] = (
    target_distribution["구×월"]
    / len(target_market_df)
    * 100
)

display(
    target_distribution.round(2)
)

,구×월,비율_%
시장방향,,
하락,389,28.29
보합,473,34.40
상승,513,37.31


In [17]:
# ============================================================
# 연도별 Target 분포
# ============================================================

target_market_df["기준년도"] = (
    target_market_df["기준월"]
    .dt.year
)

year_direction = pd.crosstab(
    target_market_df["기준년도"],
    target_market_df[CLASS_TARGET],
    normalize="index",
) * 100

year_direction = (
    year_direction
    .rename(columns=direction_name_map)
)

display(
    year_direction.round(1)
)

시장방향,하락,보합,상승
기준년도,,,
2021,60.0,26.0,14.0
2022,45.3,22.3,32.3
2023,27.7,38.0,34.3
2024,13.7,39.3,47.0
2025,19.7,41.3,39.0
2026,13.3,32.0,54.7


2025년 6월 행의 Feature는 2025년 6월 말까지 알 수 있는 정보만 사용한다.  
Target만 2025년 7~9월을 사용한다.

In [18]:
# ============================================================
# 구 × 월 과거 시장 Feature 생성
# ============================================================

market_feature_df = (
    gu_month_df
    .sort_values(
        ["구", "기준월"]
    )
    .copy()
)


grouped = market_feature_df.groupby("구")


# ------------------------------------------------------------
# 가격 Lag
# ------------------------------------------------------------

market_feature_df["평당가_lag1"] = (
    grouped["월중위평당가"]
    .shift(1)
)

market_feature_df["평당가_lag3"] = (
    grouped["월중위평당가"]
    .shift(3)
)

market_feature_df["평당가_lag6"] = (
    grouped["월중위평당가"]
    .shift(6)
)


# ------------------------------------------------------------
# 과거 가격 변화율
# ------------------------------------------------------------

market_feature_df["평당가_1개월변화율"] = (
    market_feature_df["월중위평당가"]
    / market_feature_df["평당가_lag1"]
    - 1
)

market_feature_df["평당가_3개월변화율"] = (
    market_feature_df["월중위평당가"]
    / market_feature_df["평당가_lag3"]
    - 1
)

market_feature_df["평당가_6개월변화율"] = (
    market_feature_df["월중위평당가"]
    / market_feature_df["평당가_lag6"]
    - 1
)

In [19]:
# ============================================================
# Rolling 가격 / 거래량 Feature
# ============================================================

market_feature_df["최근3개월_중위평당가"] = (
    grouped["월중위평당가"]
    .transform(
        lambda x: x.rolling(
            3,
            min_periods=3,
        ).median()
    )
)

market_feature_df["최근6개월_중위평당가"] = (
    grouped["월중위평당가"]
    .transform(
        lambda x: x.rolling(
            6,
            min_periods=6,
        ).median()
    )
)


# 최근 가격 변동성
market_feature_df["최근6개월_가격변동성"] = (
    grouped["월중위평당가"]
    .transform(
        lambda x: x.rolling(
            6,
            min_periods=6,
        ).std()
    )
)


# 거래량
market_feature_df["거래건수_lag1"] = (
    grouped["거래건수"]
    .shift(1)
)

market_feature_df["최근3개월_평균거래량"] = (
    grouped["거래건수"]
    .transform(
        lambda x: x.rolling(
            3,
            min_periods=3,
        ).mean()
    )
)

market_feature_df["최근6개월_평균거래량"] = (
    grouped["거래건수"]
    .transform(
        lambda x: x.rolling(
            6,
            min_periods=6,
        ).mean()
    )
)

In [20]:
# ============================================================
# 거래량 변화 Feature
# ============================================================

market_feature_df["거래량_전월대비"] = (
    market_feature_df["거래건수"]
    / market_feature_df["거래건수_lag1"]
    - 1
)

market_feature_df["거래량_3개월평균대비"] = (
    market_feature_df["거래건수"]
    / market_feature_df["최근3개월_평균거래량"]
    - 1
)

In [21]:
# ============================================================
# Classification 추가 Feature 후보 탐색
# ============================================================

feature_keywords = [
    "전세",
    "금리",
    "정책",
    "기준금리",
    "매매",
]

candidate_feature_cols = [
    col
    for col in df.columns
    if any(
        keyword in col
        for keyword in feature_keywords
    )
]


print(
    "추가 Feature 후보:",
    len(candidate_feature_cols),
)

for col in candidate_feature_cols:
    print("-", col)

추가 Feature 후보: 21
- 전세평당가_전월
- 전세평당가_과거3개월
- 전세평당가_과거6개월
- 전세거래량_전월
- 전세거래량_과거3개월
- 전세거래량_과거6개월
- 전세가율_전월
- 전세가율_과거3개월
- 전세가율_과거6개월
- 전세피처출처_전월
- 전세피처출처_과거3개월
- 전세피처출처_과거6개월
- 기준금리_전월
- 기준금리_3개월변화
- 기준금리_6개월변화
- 주담대금리_전월
- 주담대금리_3개월변화
- 주담대금리_6개월변화
- 전세대출금리_전월
- 전세대출금리_3개월변화
- 전세대출금리_6개월변화


In [22]:
# ============================================================
# 시장 Feature 확인
# ============================================================

market_cols = [
    "월중위평당가",
    "평당가_lag1",
    "평당가_1개월변화율",
    "평당가_3개월변화율",
    "평당가_6개월변화율",
    "최근6개월_가격변동성",
    "거래건수",
    "거래건수_lag1",
    "최근3개월_평균거래량",
    "최근6개월_평균거래량",
    "거래량_전월대비",
]

display(
    market_feature_df[
        market_cols
    ]
    .describe()
    .T
    .round(3)
)

,count,mean,std,min,25%,50%,75%,max
월중위평당가,1500.0,4562.261,1702.008,1377.417,3305.614,4102.972,5363.848,12860.399
평당가_lag1,1475.0,4548.519,1689.111,1377.417,3301.408,4095.411,5349.865,12860.399
평당가_1개월변화율,1475.0,0.008,0.108,-0.602,-0.043,0.006,0.057,1.520
평당가_3개월변화율,1425.0,0.013,0.126,-0.695,-0.055,0.013,0.075,1.734
평당가_6개월변화율,1350.0,0.020,0.151,-0.656,-0.061,0.016,0.091,2.203
최근6개월_가격변동성,1375.0,318.997,239.653,20.112,155.707,254.526,411.125,1601.750
거래건수,1500.0,156.201,143.365,4.000,53.000,117.000,205.000,1018.000
거래건수_lag1,1475.0,155.271,143.427,4.000,52.000,116.000,203.000,1018.000
최근3개월_평균거래량,1450.0,154.019,128.243,6.333,56.000,117.000,213.583,878.333
최근6개월_평균거래량,1375.0,151.197,118.338,7.833,56.667,115.667,219.000,773.500


In [23]:
# ============================================================
# 미래 Target 관련 컬럼 제거
# ============================================================

future_cols = [
    "미래1개월_평당가",
    "미래2개월_평당가",
    "미래3개월_평당가",
    "향후3개월_평당가",
    "향후3개월_변화율",
]

market_feature_df = (
    market_feature_df
    .drop(
        columns=[
            col
            for col in future_cols
            if col in market_feature_df.columns
        ]
    )
)


print("미래 관련 잔여 컬럼")

print([
    col
    for col in market_feature_df.columns
    if "미래" in col or "향후" in col
])

미래 관련 잔여 컬럼
[]


In [24]:
# ============================================================
# 구 × 월 전세 시장 Feature
# ============================================================

rent_feature_cols = [
    "전세평당가_전월",
    "전세평당가_과거3개월",
    "전세평당가_과거6개월",
    "전세거래량_전월",
    "전세거래량_과거3개월",
    "전세거래량_과거6개월",
    "전세가율_전월",
    "전세가율_과거3개월",
    "전세가율_과거6개월",
]


rent_month_df = (
    df
    .groupby(
        ["구", "기준월"],
        observed=True,
    )[rent_feature_cols]
    .median()
    .reset_index()
)


print(
    "전세 Feature:",
    rent_month_df.shape,
)

display(
    rent_month_df.head()
)

전세 Feature: (1500, 11)


,구,기준월,전세평당가_전월,전세평당가_과거3개월,전세평당가_과거6개월,전세거래량_전월,전세거래량_과거3개월,전세거래량_과거6개월,전세가율_전월,전세가율_과거3개월,전세가율_과거6개월
0,강남구,2021-07,3196.255603,3094.803759,2988.983713,11.0,25.0,35.0,<NA>,NaN,NaN
1,강남구,2021-08,3117.208864,3307.882755,3325.348814,6.0,17.5,23.0,0.397757,NaN,NaN
2,강남구,2021-09,3605.600889,3409.708790,3315.772247,8.0,24.0,30.0,0.476604,0.459748,NaN
3,강남구,2021-10,3639.790491,3305.800000,3308.623879,6.0,27.5,34.0,0.475633,0.474713,0.473732
4,강남구,2021-11,3569.486818,3209.846476,3154.395350,11.0,31.5,46.0,0.374308,0.475633,0.446981


In [25]:
# ============================================================
# 금리 Feature 월별 유일값 확인
# ============================================================

rate_feature_cols = [
    "기준금리_전월",
    "기준금리_3개월변화",
    "기준금리_6개월변화",
    "주담대금리_전월",
    "주담대금리_3개월변화",
    "주담대금리_6개월변화",
    "전세대출금리_전월",
    "전세대출금리_3개월변화",
    "전세대출금리_6개월변화",
]


rate_unique_check = (
    df
    .groupby("기준월")[rate_feature_cols]
    .nunique()
    .max()
)

display(
    rate_unique_check
    .to_frame("월별_최대_유일값수")
)

,월별_최대_유일값수
기준금리_전월,1
기준금리_3개월변화,1
기준금리_6개월변화,1
주담대금리_전월,1
주담대금리_3개월변화,1
주담대금리_6개월변화,1
전세대출금리_전월,1
전세대출금리_3개월변화,1
전세대출금리_6개월변화,1


In [26]:
# ============================================================
# 월 단위 금리 Feature
# ============================================================

rate_month_df = (
    df[
        ["기준월"]
        + rate_feature_cols
    ]
    .drop_duplicates(
        subset=["기준월"]
    )
    .sort_values("기준월")
    .reset_index(drop=True)
)


print(
    "금리 Feature:",
    rate_month_df.shape,
)

display(
    rate_month_df.head()
)

금리 Feature: (60, 10)


,기준월,기준금리_전월,기준금리_3개월변화,기준금리_6개월변화,주담대금리_전월,주담대금리_3개월변화,주담대금리_6개월변화,전세대출금리_전월,전세대출금리_3개월변화,전세대출금리_6개월변화
0,2021-07,0.50,0.00,0.00,2.74,0.01,0.15,2.52,-0.05,-0.06
1,2021-08,0.50,0.00,0.00,2.81,0.08,0.18,2.64,0.07,0.05
2,2021-09,0.75,0.25,0.25,2.88,0.19,0.22,2.72,0.20,0.15
3,2021-10,0.75,0.25,0.25,3.01,0.27,0.28,2.86,0.34,0.29
4,2021-11,0.75,0.25,0.25,3.26,0.45,0.53,3.13,0.49,0.56


In [27]:
# ============================================================
# Classification Feature 통합
# ============================================================

classification_feature_df = (
    market_feature_df
    .merge(
        rent_month_df,
        on=["구", "기준월"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        rate_month_df,
        on="기준월",
        how="left",
        validate="many_to_one",
    )
)


print(
    "Feature Table:",
    classification_feature_df.shape,
)

print(
    "구 × 월 중복:",
    classification_feature_df[
        ["구", "기준월"]
    ].duplicated().sum(),
)

Feature Table: (1500, 37)
구 × 월 중복: 0


In [28]:
# ============================================================
# Classification Feature 결측 확인
# ============================================================

feature_missing = (
    classification_feature_df
    .isna()
    .sum()
)

feature_missing = (
    feature_missing[
        feature_missing > 0
    ]
    .sort_values(
        ascending=False
    )
)

display(
    feature_missing
    .to_frame("결측건수")
)

,결측건수
평당가_6개월변화율,150
평당가_lag6,150
최근6개월_평균거래량,125
최근6개월_가격변동성,125
최근6개월_중위평당가,125
평당가_lag3,75
전세가율_과거6개월,75
평당가_3개월변화율,75
전세가율_과거3개월,50
최근3개월_평당가,50


In [29]:
# ============================================================
# 중복 시장 Feature 정리
# ============================================================

duplicate_cols = [
    "최근3개월_평당가",
]

classification_feature_df = (
    classification_feature_df
    .drop(
        columns=[
            col
            for col in duplicate_cols
            if col in classification_feature_df.columns
        ]
    )
)

print(
    "최근3개월 관련:",
    [
        col
        for col in classification_feature_df.columns
        if "최근3개월" in col
    ],
)

최근3개월 관련: ['최근3개월_중위평당가', '최근3개월_평균거래량']


In [30]:
# ============================================================
# 정책 Feature 후보 확인
# ============================================================

policy_keywords = [
    "규제",
    "조정",
    "투기",
    "정책",
    "대출",
    "LTV",
    "DTI",
    "DSR",
]

policy_candidate_cols = [
    col
    for col in df.columns
    if any(
        keyword.lower() in col.lower()
        for keyword in policy_keywords
    )
]

print(
    "정책 Feature 후보:",
    len(policy_candidate_cols),
)

for col in policy_candidate_cols:
    print("-", col)

정책 Feature 후보: 7
- 전세대출금리_전월
- 전세대출금리_3개월변화
- 전세대출금리_6개월변화
- DSR단계
- 스트레스DSR단계
- 규제지역여부
- 주담대규제강화여부


In [31]:
# ============================================================
# 정책 Feature 구조 확인
# ============================================================

policy_feature_cols = [
    "DSR단계",
    "스트레스DSR단계",
    "규제지역여부",
    "주담대규제강화여부",
]


print("===== dtype =====")

display(
    df[policy_feature_cols]
    .dtypes
    .to_frame("dtype")
)


print("\n===== 결측치 =====")

display(
    df[policy_feature_cols]
    .isna()
    .sum()
    .to_frame("결측건수")
)


print("\n===== 고유값 =====")

for col in policy_feature_cols:
    print(
        f"\n[{col}]",
        df[col]
        .dropna()
        .unique()[:20]
    )

===== dtype =====


,dtype
DSR단계,int64
스트레스DSR단계,int64
규제지역여부,int64
주담대규제강화여부,int64



===== 결측치 =====


,결측건수
DSR단계,0
스트레스DSR단계,0
규제지역여부,0
주담대규제강화여부,0



===== 고유값 =====

[DSR단계] [1 2 3]

[스트레스DSR단계] [0 1 2 3]

[규제지역여부] [1 0]

[주담대규제강화여부] [0 1]


In [32]:
# ============================================================
# 정책 Feature 단위 확인
# ============================================================

policy_granularity = []

for col in policy_feature_cols:

    month_max_unique = (
        df
        .groupby("기준월")[col]
        .nunique()
        .max()
    )

    gu_month_max_unique = (
        df
        .groupby(
            ["구", "기준월"],
            observed=True,
        )[col]
        .nunique()
        .max()
    )

    policy_granularity.append({
        "Feature": col,
        "월별_최대유일값": month_max_unique,
        "구월별_최대유일값": gu_month_max_unique,
    })


policy_granularity_df = pd.DataFrame(
    policy_granularity
)

display(policy_granularity_df)

,Feature,월별_최대유일값,구월별_최대유일값
0,DSR단계,1,1
1,스트레스DSR단계,1,1
2,규제지역여부,2,1
3,주담대규제강화여부,1,1


In [33]:
# ============================================================
# 정책 Feature 생성
# ============================================================

monthly_policy_cols = [
    "DSR단계",
    "스트레스DSR단계",
    "주담대규제강화여부",
]

policy_month_df = (
    df
    .groupby("기준월")[monthly_policy_cols]
    .first()
    .reset_index()
)


regulation_gu_month_df = (
    df
    .groupby(
        ["구", "기준월"],
        observed=True,
    )["규제지역여부"]
    .first()
    .reset_index()
)


print("월 정책:", policy_month_df.shape)
print("구×월 규제지역:", regulation_gu_month_df.shape)

월 정책: (60, 4)
구×월 규제지역: (1500, 3)


In [34]:
# ============================================================
# 정책 Feature 결합
# ============================================================

classification_feature_df = (
    classification_feature_df
    .merge(
        policy_month_df,
        on="기준월",
        how="left",
        validate="many_to_one",
    )
    .merge(
        regulation_gu_month_df,
        on=["구", "기준월"],
        how="left",
        validate="one_to_one",
    )
)


print(
    "최종 Feature Table:",
    classification_feature_df.shape,
)

print(
    "구×월 중복:",
    classification_feature_df[
        ["구", "기준월"]
    ].duplicated().sum(),
)

print(
    "정책 결측:",
    classification_feature_df[
        monthly_policy_cols
        + ["규제지역여부"]
    ]
    .isna()
    .sum()
    .sum(),
)

최종 Feature Table: (1500, 40)
구×월 중복: 0
정책 결측: 0


In [35]:
# ============================================================
# Feature + Target 결합
# ============================================================

target_cols = [
    "구",
    "기준월",
    "시장방향",
    "향후3개월_변화율",
]

classification_model_df = (
    classification_feature_df
    .merge(
        target_market_df[target_cols],
        on=["구", "기준월"],
        how="inner",
        validate="one_to_one",
    )
    .sort_values(
        ["기준월", "구"]
    )
    .reset_index(drop=True)
)


print(
    "모델링 데이터:",
    classification_model_df.shape,
)

print(
    "기간:",
    classification_model_df["기준월"].min(),
    "~",
    classification_model_df["기준월"].max(),
)

print(
    "Target 결측:",
    classification_model_df[
        "시장방향"
    ].isna().sum(),
)

모델링 데이터: (1375, 42)
기간: 2021-09 ~ 2026-03
Target 결측: 0


In [36]:
# ============================================================
# 시간순 Train / Valid / Test
# ============================================================

train_df = classification_model_df[
    classification_model_df["기준월"]
    <= pd.Period("2024-12", freq="M")
].copy()

valid_df = classification_model_df[
    classification_model_df["기준월"].between(
        pd.Period("2025-04", freq="M"),
        pd.Period("2025-09", freq="M"),
    )
].copy()

test_df = classification_model_df[
    classification_model_df["기준월"].between(
        pd.Period("2026-01", freq="M"),
        pd.Period("2026-03", freq="M"),
    )
].copy()


print("Train:", len(train_df))
print("Valid:", len(valid_df))
print("Test :", len(test_df))

Train: 1000
Valid: 150
Test : 75


In [37]:
# ============================================================
# Split별 Target 분포
# ============================================================

def class_ratio(data):
    return (
        data["시장방향"]
        .value_counts(normalize=True)
        .sort_index()
        .rename(index=direction_name_map)
        * 100
    )


split_ratio = pd.concat(
    {
        "Train": class_ratio(train_df),
        "Valid": class_ratio(valid_df),
        "Test": class_ratio(test_df),
    },
    axis=1,
)

display(split_ratio.round(1))

,Train,Valid,Test
시장방향,,,
하락,32.0,20.0,13.3
보합,32.5,43.3,32.0
상승,35.5,36.7,54.7


In [38]:
# ============================================================
# 모델 Feature 확정 + 누수 검사
# ============================================================

exclude_cols = [
    "기준월",
    "시장방향",
    "향후3개월_변화율",
]

feature_cols = [
    col
    for col in classification_model_df.columns
    if col not in exclude_cols
]

leak_cols = [
    col
    for col in feature_cols
    if (
        "미래" in col
        or "향후" in col
        or "시장방향" in col
    )
]


categorical_cols = ["구"]

numeric_cols = [
    col
    for col in feature_cols
    if col not in categorical_cols
]


print("전체 Feature:", len(feature_cols))
print("수치형:", len(numeric_cols))
print("범주형:", len(categorical_cols))
print("누수 의심:", leak_cols)

전체 Feature: 39
수치형: 38
범주형: 1
누수 의심: []


In [39]:
# ============================================================
# Train / Valid / Test 데이터 준비
# ============================================================

X_train_raw = train_df[feature_cols]
X_valid_raw = valid_df[feature_cols]
X_test_raw = test_df[feature_cols]

y_train = train_df["시장방향"]
y_valid = valid_df["시장방향"]
y_test = test_df["시장방향"]


print("Train:", X_train_raw.shape)
print("Valid:", X_valid_raw.shape)
print("Test :", X_test_raw.shape)

Train: (1000, 39)
Valid: (150, 39)
Test : (75, 39)


In [40]:
# ============================================================
# 구 One-Hot Encoding
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder


preprocessor = ColumnTransformer(
    transformers=[
        (
            "gu",
            OneHotEncoder(
                handle_unknown="ignore",
            ),
            categorical_cols,
        ),
    ],
    remainder="passthrough",
)


X_train = preprocessor.fit_transform(
    X_train_raw
)

X_valid = preprocessor.transform(
    X_valid_raw
)

X_test = preprocessor.transform(
    X_test_raw
)


print("변환 후 Train:", X_train.shape)
print("변환 후 Valid:", X_valid.shape)
print("변환 후 Test :", X_test.shape)

변환 후 Train: (1000, 63)
변환 후 Valid: (150, 63)
변환 후 Test : (75, 63)


In [41]:
# ============================================================
# XGBoost Classification V1 Baseline
# ============================================================

from xgboost import XGBClassifier


classification_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,

    n_estimators=3000,
    learning_rate=0.03,

    max_depth=2,
    min_child_weight=5,

    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=0.2,
    reg_lambda=5.0,

    tree_method="hist",
    eval_metric="mlogloss",
    early_stopping_rounds=100,

    random_state=42,
    n_jobs=-1,
)


classification_model.fit(
    X_train,
    y_train,
    eval_set=[
        (X_valid, y_valid),
    ],
    verbose=False,
)


print(
    "Best iteration:",
    classification_model.best_iteration,
)

Best iteration: 0


In [42]:
# ============================================================
# Validation 평가
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    classification_report,
)


valid_pred = classification_model.predict(
    X_valid
)

valid_prob = classification_model.predict_proba(
    X_valid
)


print(
    "Accuracy:",
    round(
        accuracy_score(
            y_valid,
            valid_pred,
        ),
        4,
    ),
)

print(
    "Balanced Accuracy:",
    round(
        balanced_accuracy_score(
            y_valid,
            valid_pred,
        ),
        4,
    ),
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_valid,
            valid_pred,
            average="macro",
        ),
        4,
    ),
)

print(
    "Log Loss:",
    round(
        log_loss(
            y_valid,
            valid_prob,
        ),
        4,
    ),
)


print("\nClassification Report")

print(
    classification_report(
        y_valid,
        valid_pred,
        target_names=[
            "하락",
            "보합",
            "상승",
        ],
        digits=3,
        zero_division=0,
    )
)

Accuracy: 0.3667
Balanced Accuracy: 0.3333
Macro F1: 0.1789
Log Loss: 1.0956

Classification Report
              precision    recall  f1-score   support

          하락      0.000     0.000     0.000        30
          보합      0.000     0.000     0.000        65
          상승      0.367     1.000     0.537        55

    accuracy                          0.367       150
   macro avg      0.122     0.333     0.179       150
weighted avg      0.134     0.367     0.197       150



In [43]:
# ============================================================
# Majority Baseline
# ============================================================

majority_class = (
    y_train
    .value_counts()
    .idxmax()
)

majority_pred = np.full(
    len(y_valid),
    majority_class,
)


print(
    "Majority Class:",
    direction_name_map[
        majority_class
    ],
)

print(
    "Majority Accuracy:",
    round(
        accuracy_score(
            y_valid,
            majority_pred,
        ),
        4,
    ),
)

print(
    "XGBoost Accuracy:",
    round(
        accuracy_score(
            y_valid,
            valid_pred,
        ),
        4,
    ),
)

Majority Class: 상승
Majority Accuracy: 0.3667
XGBoost Accuracy: 0.3667


In [44]:
# ============================================================
# Train / Valid 학습 상태 확인
# ============================================================

train_pred = classification_model.predict(X_train)

print("Best iteration:", classification_model.best_iteration)

print("\n[Train]")
print(
    "Accuracy:",
    round(accuracy_score(y_train, train_pred), 4),
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_train,
            train_pred,
            average="macro",
        ),
        4,
    ),
)

print("\nTrain 예측 분포")
display(
    pd.Series(train_pred)
    .value_counts(normalize=True)
    .sort_index()
    .rename(index=direction_name_map)
    .mul(100)
    .to_frame("예측비율_%")
    .round(1)
)

print("\nValid 예측 분포")
display(
    pd.Series(valid_pred)
    .value_counts(normalize=True)
    .sort_index()
    .rename(index=direction_name_map)
    .mul(100)
    .to_frame("예측비율_%")
    .round(1)
)

Best iteration: 0

[Train]
Accuracy: 0.355
Macro F1: 0.1747

Train 예측 분포


,예측비율_%
상승,100.0



Valid 예측 분포


,예측비율_%
상승,100.0


In [45]:
# ============================================================
# Train / Valid 학습 상태 확인
# ============================================================

train_pred = classification_model.predict(X_train)

print("Best iteration:", classification_model.best_iteration)

print("\n[Train]")
print(
    "Accuracy:",
    round(accuracy_score(y_train, train_pred), 4),
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_train,
            train_pred,
            average="macro",
        ),
        4,
    ),
)

print("\nTrain 예측 분포")
display(
    pd.Series(train_pred)
    .value_counts(normalize=True)
    .sort_index()
    .rename(index=direction_name_map)
    .mul(100)
    .to_frame("예측비율_%")
    .round(1)
)

print("\nValid 예측 분포")
display(
    pd.Series(valid_pred)
    .value_counts(normalize=True)
    .sort_index()
    .rename(index=direction_name_map)
    .mul(100)
    .to_frame("예측비율_%")
    .round(1)
)

Best iteration: 0

[Train]
Accuracy: 0.355
Macro F1: 0.1747

Train 예측 분포


,예측비율_%
상승,100.0



Valid 예측 분포


,예측비율_%
상승,100.0


In [46]:
# ============================================================
# XGBoost 학습 가능성 진단
# ============================================================

diagnostic_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,

    n_estimators=200,
    learning_rate=0.05,

    max_depth=3,
    min_child_weight=1,

    subsample=1.0,
    colsample_bytree=1.0,

    reg_alpha=0.0,
    reg_lambda=1.0,

    tree_method="hist",
    eval_metric="mlogloss",

    random_state=42,
    n_jobs=-1,
)


_ = diagnostic_model.fit(
    X_train,
    y_train,
    verbose=False,
)

print("학습 완료")

학습 완료


In [47]:
# ============================================================
# Diagnostic Model 평가
# ============================================================

diagnostic_train_pred = diagnostic_model.predict(
    X_train
)

diagnostic_valid_pred = diagnostic_model.predict(
    X_valid
)


print("[Train]")

print(
    "Accuracy:",
    round(
        accuracy_score(
            y_train,
            diagnostic_train_pred,
        ),
        4,
    ),
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_train,
            diagnostic_train_pred,
            average="macro",
        ),
        4,
    ),
)


print("\n[Valid]")

print(
    "Accuracy:",
    round(
        accuracy_score(
            y_valid,
            diagnostic_valid_pred,
        ),
        4,
    ),
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_valid,
            diagnostic_valid_pred,
            average="macro",
        ),
        4,
    ),
)

[Train]
Accuracy: 0.851
Macro F1: 0.8499

[Valid]
Accuracy: 0.3867
Macro F1: 0.254


In [48]:
# ============================================================
# 예측 클래스 분포
# ============================================================

for name, pred in [
    ("Train", diagnostic_train_pred),
    ("Valid", diagnostic_valid_pred),
]:

    pred_ratio = (
        pd.Series(pred)
        .value_counts(normalize=True)
        .sort_index()
        .rename(index=direction_name_map)
        * 100
    )

    print(f"\n[{name}]")

    display(
        pred_ratio
        .to_frame("예측비율_%")
        .round(1)
    )


[Train]


,예측비율_%
하락,35.4
보합,29.3
상승,35.3



[Valid]


,예측비율_%
하락,3.3
보합,72.0
상승,24.7


In [49]:
# ============================================================
# Diagnostic Model 성능 정리
# ============================================================

for name, y_true, y_pred in [
    ("Train", y_train, diagnostic_train_pred),
    ("Valid", y_valid, diagnostic_valid_pred),
]:
    print(f"[{name}]")

    print(
        "Accuracy:",
        round(
            accuracy_score(y_true, y_pred),
            4,
        ),
    )

    print(
        "Balanced Accuracy:",
        round(
            balanced_accuracy_score(
                y_true,
                y_pred,
            ),
            4,
        ),
    )

    print(
        "Macro F1:",
        round(
            f1_score(
                y_true,
                y_pred,
                average="macro",
            ),
            4,
        ),
    )

    print()

[Train]
Accuracy: 0.851
Balanced Accuracy: 0.8505
Macro F1: 0.8499

[Valid]
Accuracy: 0.3867
Balanced Accuracy: 0.3058
Macro F1: 0.254



In [50]:
# ============================================================
# Validation Confusion Matrix
# ============================================================

from sklearn.metrics import confusion_matrix

valid_cm = confusion_matrix(
    y_valid,
    diagnostic_valid_pred,
    labels=[0, 1, 2],
)

valid_cm_df = pd.DataFrame(
    valid_cm,
    index=[
        "실제_하락",
        "실제_보합",
        "실제_상승",
    ],
    columns=[
        "예측_하락",
        "예측_보합",
        "예측_상승",
    ],
)

display(valid_cm_df)

,예측_하락,예측_보합,예측_상승
실제_하락,0,15,15
실제_보합,3,49,13
실제_상승,2,44,9


In [51]:
# ============================================================
# Validation 월별 실제 / 예측 분포
# ============================================================

valid_result_df = valid_df[
    ["구", "기준월", "시장방향"]
].copy()

valid_result_df["예측방향"] = (
    diagnostic_valid_pred
)


monthly_result = []

for month, month_df in valid_result_df.groupby("기준월"):

    actual = (
        month_df["시장방향"]
        .value_counts(normalize=True)
        .reindex([0, 1, 2], fill_value=0)
        * 100
    )

    pred = (
        month_df["예측방향"]
        .value_counts(normalize=True)
        .reindex([0, 1, 2], fill_value=0)
        * 100
    )

    monthly_result.append({
        "기준월": month,
        "실제_하락": actual[0],
        "실제_보합": actual[1],
        "실제_상승": actual[2],
        "예측_하락": pred[0],
        "예측_보합": pred[1],
        "예측_상승": pred[2],
    })


monthly_result_df = pd.DataFrame(
    monthly_result
)

display(
    monthly_result_df.round(1)
)

,기준월,실제_하락,실제_보합,실제_상승,예측_하락,예측_보합,예측_상승
0,2025-04,16.0,56.0,28.0,4.0,64.0,32.0
1,2025-05,40.0,44.0,16.0,0.0,80.0,20.0
2,2025-06,40.0,48.0,12.0,0.0,60.0,40.0
3,2025-07,16.0,28.0,56.0,0.0,92.0,8.0
4,2025-08,0.0,32.0,68.0,12.0,68.0,20.0
5,2025-09,8.0,52.0,40.0,4.0,68.0,28.0


In [52]:
# ============================================================
# 서울 전체 시장 흐름 Feature
# ============================================================

seoul_market_df = (
    gu_month_df
    .groupby("기준월")
    .agg(
        서울_중위평당가=(
            "월중위평당가",
            "median",
        ),
        서울_거래건수=(
            "거래건수",
            "sum",
        ),
    )
    .sort_index()
    .reset_index()
)


seoul_market_df["서울_평당가_lag1"] = (
    seoul_market_df["서울_중위평당가"]
    .shift(1)
)

seoul_market_df["서울_평당가_lag3"] = (
    seoul_market_df["서울_중위평당가"]
    .shift(3)
)

seoul_market_df["서울_평당가_lag6"] = (
    seoul_market_df["서울_중위평당가"]
    .shift(6)
)


seoul_market_df["서울_1개월변화율"] = (
    seoul_market_df["서울_중위평당가"]
    / seoul_market_df["서울_평당가_lag1"]
    - 1
)

seoul_market_df["서울_3개월변화율"] = (
    seoul_market_df["서울_중위평당가"]
    / seoul_market_df["서울_평당가_lag3"]
    - 1
)

seoul_market_df["서울_6개월변화율"] = (
    seoul_market_df["서울_중위평당가"]
    / seoul_market_df["서울_평당가_lag6"]
    - 1
)

In [53]:
# ============================================================
# 서울 시장 Breadth Feature
# ============================================================

breadth_df = (
    market_feature_df
    .groupby("기준월")
    .agg(
        상승구비율=(
            "평당가_3개월변화율",
            lambda x: (x > 0).mean(),
        ),
        강한상승구비율=(
            "평당가_3개월변화율",
            lambda x: (x > 0.03).mean(),
        ),
        강한하락구비율=(
            "평당가_3개월변화율",
            lambda x: (x < -0.03).mean(),
        ),
    )
    .reset_index()
)


seoul_market_df = (
    seoul_market_df
    .merge(
        breadth_df,
        on="기준월",
        how="left",
        validate="one_to_one",
    )
)

In [54]:
# ============================================================
# 직전 확정 3개월 시장 변화
# ============================================================

confirmed_market_df = (
    gu_month_df[
        [
            "구",
            "기준월",
            "향후3개월_변화율",
        ]
    ]
    .sort_values(
        ["구", "기준월"]
    )
    .copy()
)


confirmed_market_df["직전확정_3개월변화율"] = (
    confirmed_market_df
    .groupby("구")[
        "향후3개월_변화율"
    ]
    .shift(3)
)


confirmed_market_df = (
    confirmed_market_df[
        [
            "구",
            "기준월",
            "직전확정_3개월변화율",
        ]
    ]
)

In [55]:
# ============================================================
# 시장 국면 Feature 결합
# ============================================================

classification_feature_v1_df = (
    classification_feature_df
    .merge(
        seoul_market_df,
        on="기준월",
        how="left",
        validate="many_to_one",
    )
    .merge(
        confirmed_market_df,
        on=["구", "기준월"],
        how="left",
        validate="one_to_one",
    )
)


print(
    "Feature Table:",
    classification_feature_v1_df.shape,
)

print(
    "구×월 중복:",
    classification_feature_v1_df[
        ["구", "기준월"]
    ].duplicated().sum(),
)

Feature Table: (1500, 52)
구×월 중복: 0


In [56]:
# ============================================================
# 시장 국면 Feature + Target 결합
# ============================================================

classification_model_v1_df = (
    classification_feature_v1_df
    .merge(
        target_market_df[
            [
                "구",
                "기준월",
                "시장방향",
                "향후3개월_변화율",
            ]
        ],
        on=["구", "기준월"],
        how="inner",
        validate="one_to_one",
    )
    .sort_values(
        ["기준월", "구"]
    )
    .reset_index(drop=True)
)


print(
    "모델링 데이터:",
    classification_model_v1_df.shape,
)

print(
    "기간:",
    classification_model_v1_df["기준월"].min(),
    "~",
    classification_model_v1_df["기준월"].max(),
)

모델링 데이터: (1375, 54)
기간: 2021-09 ~ 2026-03


In [57]:
# ============================================================
# 동일한 Train / Valid / Test 분할
# ============================================================

train_v1_df = classification_model_v1_df[
    classification_model_v1_df["기준월"]
    <= pd.Period("2024-12", freq="M")
].copy()

valid_v1_df = classification_model_v1_df[
    classification_model_v1_df["기준월"].between(
        pd.Period("2025-04", freq="M"),
        pd.Period("2025-09", freq="M"),
    )
].copy()

test_v1_df = classification_model_v1_df[
    classification_model_v1_df["기준월"].between(
        pd.Period("2026-01", freq="M"),
        pd.Period("2026-03", freq="M"),
    )
].copy()


print("Train:", len(train_v1_df))
print("Valid:", len(valid_v1_df))
print("Test :", len(test_v1_df))

Train: 1000
Valid: 150
Test : 75


In [58]:
# ============================================================
# Feature 확정 + 누수 검사
# ============================================================

exclude_cols = [
    "기준월",
    "시장방향",
    "향후3개월_변화율",
    "상승여부",  # Binary Target 재실행 시 Feature 유입 방지
]

feature_v1_cols = [
    col
    for col in classification_model_v1_df.columns
    if col not in exclude_cols
]

categorical_v1_cols = ["구"]

numeric_v1_cols = [
    col
    for col in feature_v1_cols
    if col not in categorical_v1_cols
]


leak_cols = [
    col
    for col in feature_v1_cols
    if (
        "미래" in col
        or "향후" in col
        or "시장방향" in col
    )
]


print("전체 Feature:", len(feature_v1_cols))
print("수치형:", len(numeric_v1_cols))
print("누수 의심:", leak_cols)

전체 Feature: 51
수치형: 50
누수 의심: []


In [59]:
# ============================================================
# X / y + One-Hot Encoding
# ============================================================

X_train_v1_raw = train_v1_df[feature_v1_cols]
X_valid_v1_raw = valid_v1_df[feature_v1_cols]

y_train_v1 = train_v1_df["시장방향"]
y_valid_v1 = valid_v1_df["시장방향"]


preprocessor_v1 = ColumnTransformer(
    transformers=[
        (
            "gu",
            OneHotEncoder(
                handle_unknown="ignore",
            ),
            categorical_v1_cols,
        ),
    ],
    remainder="passthrough",
)


X_train_v1 = preprocessor_v1.fit_transform(
    X_train_v1_raw
)

X_valid_v1 = preprocessor_v1.transform(
    X_valid_v1_raw
)


print("Train:", X_train_v1.shape)
print("Valid:", X_valid_v1.shape)

Train: (1000, 75)
Valid: (150, 75)


In [60]:
# ============================================================
# Enhanced Feature Diagnostic Model
# ============================================================

diagnostic_v1_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,

    n_estimators=200,
    learning_rate=0.05,

    max_depth=3,
    min_child_weight=1,

    subsample=1.0,
    colsample_bytree=1.0,

    reg_alpha=0.0,
    reg_lambda=1.0,

    tree_method="hist",
    eval_metric="mlogloss",

    random_state=42,
    n_jobs=-1,
)


_ = diagnostic_v1_model.fit(
    X_train_v1,
    y_train_v1,
    verbose=False,
)

In [61]:
# ============================================================
# 기존 / 개선 Feature 성능 비교
# ============================================================

train_v1_pred = diagnostic_v1_model.predict(
    X_train_v1
)

valid_v1_pred = diagnostic_v1_model.predict(
    X_valid_v1
)


comparison = pd.DataFrame(
    [
        {
            "모델": "기존 Feature",
            "Train Accuracy": accuracy_score(
                y_train,
                diagnostic_train_pred,
            ),
            "Train Macro F1": f1_score(
                y_train,
                diagnostic_train_pred,
                average="macro",
            ),
            "Valid Accuracy": accuracy_score(
                y_valid,
                diagnostic_valid_pred,
            ),
            "Valid Macro F1": f1_score(
                y_valid,
                diagnostic_valid_pred,
                average="macro",
            ),
        },
        {
            "모델": "시장국면 Feature 추가",
            "Train Accuracy": accuracy_score(
                y_train_v1,
                train_v1_pred,
            ),
            "Train Macro F1": f1_score(
                y_train_v1,
                train_v1_pred,
                average="macro",
            ),
            "Valid Accuracy": accuracy_score(
                y_valid_v1,
                valid_v1_pred,
            ),
            "Valid Macro F1": f1_score(
                y_valid_v1,
                valid_v1_pred,
                average="macro",
            ),
        },
    ]
)


display(
    comparison.round(4)
)

,모델,Train Accuracy,Train Macro F1,Valid Accuracy,Valid Macro F1
0,기존 Feature,0.851,0.8499,0.3867,0.2540
1,시장국면 Feature 추가,0.859,0.8580,0.3667,0.2453


In [62]:
print(
    classification_report(
        y_valid_v1,
        valid_v1_pred,
        target_names=[
            "하락",
            "보합",
            "상승",
        ],
        digits=3,
        zero_division=0,
    )
)

              precision    recall  f1-score   support

          하락      0.000     0.000     0.000        30
          보합      0.460     0.708     0.558        65
          상승      0.196     0.164     0.178        55

    accuracy                          0.367       150
   macro avg      0.219     0.290     0.245       150
weighted avg      0.271     0.367     0.307       150



In [63]:
from sklearn.metrics import recall_score

In [64]:
# ============================================================
# Expanding Rolling Backtest 구간
# ============================================================

backtest_periods = [
    {
        "name": "Fold1",
        "train_end": "2023-06",
        "valid_start": "2023-10",
        "valid_end": "2024-03",
    },
    {
        "name": "Fold2",
        "train_end": "2023-12",
        "valid_start": "2024-04",
        "valid_end": "2024-09",
    },
    {
        "name": "Fold3",
        "train_end": "2024-06",
        "valid_start": "2024-10",
        "valid_end": "2025-03",
    },
    {
        "name": "Fold4",
        "train_end": "2024-12",
        "valid_start": "2025-04",
        "valid_end": "2025-09",
    },
]

In [65]:
# ============================================================
# Rolling Backtest
# ============================================================

backtest_results = []

for period in backtest_periods:

    train_end = pd.Period(
        period["train_end"],
        freq="M",
    )

    valid_start = pd.Period(
        period["valid_start"],
        freq="M",
    )

    valid_end = pd.Period(
        period["valid_end"],
        freq="M",
    )


    fold_train = classification_model_v1_df[
        classification_model_v1_df["기준월"]
        <= train_end
    ].copy()

    fold_valid = classification_model_v1_df[
        classification_model_v1_df["기준월"]
        .between(
            valid_start,
            valid_end,
        )
    ].copy()


    X_fold_train_raw = fold_train[
        feature_v1_cols
    ]

    X_fold_valid_raw = fold_valid[
        feature_v1_cols
    ]

    y_fold_train = fold_train["시장방향"]
    y_fold_valid = fold_valid["시장방향"]


    fold_preprocessor = ColumnTransformer(
        transformers=[
            (
                "gu",
                OneHotEncoder(
                    handle_unknown="ignore"
                ),
                ["구"],
            ),
        ],
        remainder="passthrough",
    )


    X_fold_train = (
        fold_preprocessor
        .fit_transform(X_fold_train_raw)
    )

    X_fold_valid = (
        fold_preprocessor
        .transform(X_fold_valid_raw)
    )


    fold_model = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        min_child_weight=1,
        subsample=1.0,
        colsample_bytree=1.0,
        reg_alpha=0.0,
        reg_lambda=1.0,
        tree_method="hist",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
    )


    _ = fold_model.fit(
        X_fold_train,
        y_fold_train,
        verbose=False,
    )


    fold_pred = fold_model.predict(
        X_fold_valid
    )


    backtest_results.append({
        "Fold": period["name"],
        "Train": len(fold_train),
        "Valid": len(fold_valid),
        "Accuracy": accuracy_score(
            y_fold_valid,
            fold_pred,
        ),
        "Balanced Accuracy":
            balanced_accuracy_score(
                y_fold_valid,
                fold_pred,
            ),
        "Macro F1": f1_score(
            y_fold_valid,
            fold_pred,
            average="macro",
        ),
        "하락 Recall": (
            recall_score(
                y_fold_valid,
                fold_pred,
                labels=[0],
                average="macro",
                zero_division=0,
            )
        ),
        "보합 Recall": (
            recall_score(
                y_fold_valid,
                fold_pred,
                labels=[1],
                average="macro",
                zero_division=0,
            )
        ),
        "상승 Recall": (
            recall_score(
                y_fold_valid,
                fold_pred,
                labels=[2],
                average="macro",
                zero_division=0,
            )
        ),
    })


backtest_df = pd.DataFrame(
    backtest_results
)

display(
    backtest_df.round(3)
)

,Fold,Train,Valid,Accuracy,Balanced Accuracy,Macro F1,하락 Recall,보합 Recall,상승 Recall
0,Fold1,550,150,0.467,0.408,0.377,0.103,0.306,0.814
1,Fold2,700,150,0.413,0.401,0.380,0.235,0.698,0.270
2,Fold3,850,150,0.393,0.459,0.357,0.600,0.538,0.240
3,Fold4,1000,150,0.367,0.290,0.245,0.000,0.708,0.164


In [66]:
# ============================================================
# Binary Target
# 0 = 비상승
# 1 = 상승(+3% 초과)
# ============================================================

BINARY_TARGET = "상승여부"

classification_model_v1_df[BINARY_TARGET] = (
    classification_model_v1_df[
        "향후3개월_변화율"
    ] > 0.03
).astype("int8")


binary_distribution = (
    classification_model_v1_df[BINARY_TARGET]
    .value_counts()
    .sort_index()
    .rename(
        index={
            0: "비상승",
            1: "상승",
        }
    )
    .to_frame("구×월")
)

binary_distribution["비율_%"] = (
    binary_distribution["구×월"]
    / len(classification_model_v1_df)
    * 100
)

display(binary_distribution.round(2))

,구×월,비율_%
상승여부,,
비상승,862,62.69
상승,513,37.31


In [67]:
# ============================================================
# 연도별 Binary Target 분포
# ============================================================

binary_check_df = (
    classification_model_v1_df[
        ["기준월", BINARY_TARGET]
    ]
    .copy()
)

binary_check_df["기준년도"] = (
    binary_check_df["기준월"].dt.year
)


year_binary_ratio = pd.crosstab(
    binary_check_df["기준년도"],
    binary_check_df[BINARY_TARGET],
    normalize="index",
) * 100


year_binary_ratio = year_binary_ratio.rename(
    columns={
        0: "비상승",
        1: "상승",
    }
)


display(
    year_binary_ratio.round(1)
)

상승여부,비상승,상승
기준년도,,
2021,86.0,14.0
2022,67.7,32.3
2023,65.7,34.3
2024,53.0,47.0
2025,61.0,39.0
2026,45.3,54.7


In [68]:
# ============================================================
# Binary Target Split별 분포
# ============================================================

def binary_ratio(data):
    return (
        data[BINARY_TARGET]
        .value_counts(normalize=True)
        .sort_index()
        .rename({
            0: "비상승",
            1: "상승",
        })
        * 100
    )


train_binary_df = classification_model_v1_df[
    classification_model_v1_df["기준월"]
    <= pd.Period("2024-12", freq="M")
].copy()

valid_binary_df = classification_model_v1_df[
    classification_model_v1_df["기준월"].between(
        pd.Period("2025-04", freq="M"),
        pd.Period("2025-09", freq="M"),
    )
].copy()

test_binary_df = classification_model_v1_df[
    classification_model_v1_df["기준월"].between(
        pd.Period("2026-01", freq="M"),
        pd.Period("2026-03", freq="M"),
    )
].copy()


split_binary_ratio = pd.concat(
    {
        "Train": binary_ratio(train_binary_df),
        "Valid": binary_ratio(valid_binary_df),
        "Test": binary_ratio(test_binary_df),
    },
    axis=1,
)

display(split_binary_ratio.round(1))

,Train,Valid,Test
상승여부,,,
비상승,64.5,63.3,45.3
상승,35.5,36.7,54.7


In [69]:
# ============================================================
# Binary XGBoost Rolling Backtest
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

binary_backtest_results = []


for period in backtest_periods:

    train_end = pd.Period(
        period["train_end"],
        freq="M",
    )

    valid_start = pd.Period(
        period["valid_start"],
        freq="M",
    )

    valid_end = pd.Period(
        period["valid_end"],
        freq="M",
    )


    fold_train = classification_model_v1_df[
        classification_model_v1_df["기준월"]
        <= train_end
    ].copy()

    fold_valid = classification_model_v1_df[
        classification_model_v1_df["기준월"]
        .between(
            valid_start,
            valid_end,
        )
    ].copy()


    X_fold_train_raw = fold_train[
        feature_v1_cols
    ]

    X_fold_valid_raw = fold_valid[
        feature_v1_cols
    ]

    y_fold_train = fold_train[BINARY_TARGET]
    y_fold_valid = fold_valid[BINARY_TARGET]


    fold_preprocessor = ColumnTransformer(
        transformers=[
            (
                "gu",
                OneHotEncoder(
                    handle_unknown="ignore"
                ),
                ["구"],
            ),
        ],
        remainder="passthrough",
    )


    X_fold_train = (
        fold_preprocessor
        .fit_transform(X_fold_train_raw)
    )

    X_fold_valid = (
        fold_preprocessor
        .transform(X_fold_valid_raw)
    )


    fold_model = XGBClassifier(
        objective="binary:logistic",

        n_estimators=200,
        learning_rate=0.05,

        max_depth=3,
        min_child_weight=1,

        subsample=1.0,
        colsample_bytree=1.0,

        reg_alpha=0.0,
        reg_lambda=1.0,

        tree_method="hist",
        eval_metric="logloss",

        random_state=42,
        n_jobs=-1,
    )


    _ = fold_model.fit(
        X_fold_train,
        y_fold_train,
        verbose=False,
    )


    fold_pred = fold_model.predict(
        X_fold_valid
    )

    fold_prob = fold_model.predict_proba(
        X_fold_valid
    )[:, 1]


    binary_backtest_results.append({
        "Fold": period["name"],
        "Train": len(fold_train),
        "Valid": len(fold_valid),

        "Accuracy": accuracy_score(
            y_fold_valid,
            fold_pred,
        ),

        "Balanced Accuracy":
            balanced_accuracy_score(
                y_fold_valid,
                fold_pred,
            ),

        "F1": f1_score(
            y_fold_valid,
            fold_pred,
        ),

        "Precision": precision_score(
            y_fold_valid,
            fold_pred,
            zero_division=0,
        ),

        "상승 Recall": recall_score(
            y_fold_valid,
            fold_pred,
            zero_division=0,
        ),

        "ROC AUC": roc_auc_score(
            y_fold_valid,
            fold_prob,
        ),
    })


binary_backtest_df = pd.DataFrame(
    binary_backtest_results
)

display(
    binary_backtest_df.round(3)
)

,Fold,Train,Valid,Accuracy,Balanced Accuracy,F1,Precision,상승 Recall,ROC AUC
0,Fold1,550,150,0.633,0.647,0.604,0.525,0.712,0.707
1,Fold2,700,150,0.667,0.612,0.405,0.810,0.270,0.679
2,Fold3,850,150,0.473,0.473,0.151,0.389,0.093,0.524
3,Fold4,1000,150,0.507,0.415,0.098,0.148,0.073,0.307


In [70]:
# ============================================================
# 최근 N개월 Rolling Window 후보
# ============================================================

rolling_windows = [
    18,
    24,
    30,
    36,
]

In [71]:
# ============================================================
# 최근 N개월 Rolling Window Backtest
# ============================================================

rolling_results = []


for window_months in rolling_windows:

    for period in backtest_periods:

        train_end = pd.Period(
            period["train_end"],
            freq="M",
        )

        valid_start = pd.Period(
            period["valid_start"],
            freq="M",
        )

        valid_end = pd.Period(
            period["valid_end"],
            freq="M",
        )

        train_start = (
            train_end
            - (window_months - 1)
        )


        fold_train = classification_model_v1_df[
            classification_model_v1_df[
                "기준월"
            ].between(
                train_start,
                train_end,
            )
        ].copy()

        fold_valid = classification_model_v1_df[
            classification_model_v1_df[
                "기준월"
            ].between(
                valid_start,
                valid_end,
            )
        ].copy()


        X_fold_train_raw = fold_train[
            feature_v1_cols
        ]

        X_fold_valid_raw = fold_valid[
            feature_v1_cols
        ]

        y_fold_train = fold_train[
            BINARY_TARGET
        ]

        y_fold_valid = fold_valid[
            BINARY_TARGET
        ]


        fold_preprocessor = ColumnTransformer(
            transformers=[
                (
                    "gu",
                    OneHotEncoder(
                        handle_unknown="ignore",
                    ),
                    ["구"],
                ),
            ],
            remainder="passthrough",
        )


        X_fold_train = (
            fold_preprocessor
            .fit_transform(
                X_fold_train_raw
            )
        )

        X_fold_valid = (
            fold_preprocessor
            .transform(
                X_fold_valid_raw
            )
        )


        fold_model = XGBClassifier(
            objective="binary:logistic",

            n_estimators=200,
            learning_rate=0.05,

            max_depth=3,
            min_child_weight=1,

            subsample=1.0,
            colsample_bytree=1.0,

            reg_alpha=0.0,
            reg_lambda=1.0,

            tree_method="hist",
            eval_metric="logloss",

            random_state=42,
            n_jobs=-1,
        )


        _ = fold_model.fit(
            X_fold_train,
            y_fold_train,
            verbose=False,
        )


        fold_pred = fold_model.predict(
            X_fold_valid
        )

        fold_prob = (
            fold_model
            .predict_proba(
                X_fold_valid
            )[:, 1]
        )


        rolling_results.append({
            "Window": window_months,
            "Fold": period["name"],
            "Train": len(fold_train),

            "Balanced Accuracy":
                balanced_accuracy_score(
                    y_fold_valid,
                    fold_pred,
                ),

            "F1":
                f1_score(
                    y_fold_valid,
                    fold_pred,
                    zero_division=0,
                ),

            "상승 Recall":
                recall_score(
                    y_fold_valid,
                    fold_pred,
                    zero_division=0,
                ),

            "ROC AUC":
                roc_auc_score(
                    y_fold_valid,
                    fold_prob,
                ),
        })


rolling_backtest_df = pd.DataFrame(
    rolling_results
)

display(
    rolling_backtest_df.round(3)
)

,Window,Fold,Train,Balanced Accuracy,F1,상승 Recall,ROC AUC
0,18,Fold1,450,0.678,0.632,0.729,0.742
1,18,Fold2,450,0.572,0.304,0.190,0.720
2,18,Fold3,450,0.460,0.214,0.147,0.567
3,18,Fold4,450,0.482,0.354,0.364,0.422
4,24,Fold1,550,0.647,0.604,0.712,0.707
5,24,Fold2,600,0.589,0.316,0.190,0.730
6,24,Fold3,600,0.500,0.157,0.093,0.527
7,24,Fold4,600,0.455,0.263,0.236,0.401
8,30,Fold1,550,0.647,0.604,0.712,0.707
9,30,Fold2,700,0.612,0.405,0.270,0.679


In [72]:
# ============================================================
# Window별 평균 성능
# ============================================================

window_summary = (
    rolling_backtest_df
    .groupby("Window")
    .agg(
        평균_Balanced_Accuracy=(
            "Balanced Accuracy",
            "mean",
        ),
        평균_F1=(
            "F1",
            "mean",
        ),
        평균_상승_Recall=(
            "상승 Recall",
            "mean",
        ),
        평균_ROC_AUC=(
            "ROC AUC",
            "mean",
        ),
        최소_ROC_AUC=(
            "ROC AUC",
            "min",
        ),
    )
)

display(
    window_summary.round(3)
)

,평균_Balanced_Accuracy,평균_F1,평균_상승_Recall,평균_ROC_AUC,최소_ROC_AUC
Window,,,,,
18,0.548,0.376,0.357,0.613,0.422
24,0.548,0.335,0.308,0.591,0.401
30,0.538,0.329,0.299,0.555,0.322
36,0.533,0.313,0.287,0.558,0.321


In [73]:
# ============================================================
# 18개월 Rolling Window Fold별 성능
# ============================================================

rolling_18m_df = (
    rolling_backtest_df[
        rolling_backtest_df["Window"] == 18
    ]
    .reset_index(drop=True)
)

display(
    rolling_18m_df.round(3)
)

,Window,Fold,Train,Balanced Accuracy,F1,상승 Recall,ROC AUC
0,18,Fold1,450,0.678,0.632,0.729,0.742
1,18,Fold2,450,0.572,0.304,0.190,0.720
2,18,Fold3,450,0.460,0.214,0.147,0.567
3,18,Fold4,450,0.482,0.354,0.364,0.422


In [74]:
# ============================================================
# Expanding vs 18개월 Rolling 비교
# ============================================================

expanding_compare = (
    binary_backtest_df[
        [
            "Fold",
            "Balanced Accuracy",
            "F1",
            "상승 Recall",
            "ROC AUC",
        ]
    ]
    .copy()
)

expanding_compare["학습방식"] = "Expanding"


rolling_compare = (
    rolling_18m_df[
        [
            "Fold",
            "Balanced Accuracy",
            "F1",
            "상승 Recall",
            "ROC AUC",
        ]
    ]
    .copy()
)

rolling_compare["학습방식"] = "18개월 Rolling"


window_compare = pd.concat(
    [
        expanding_compare,
        rolling_compare,
    ],
    ignore_index=True,
)

display(
    window_compare[
        [
            "Fold",
            "학습방식",
            "Balanced Accuracy",
            "F1",
            "상승 Recall",
            "ROC AUC",
        ]
    ].round(3)
)

,Fold,학습방식,Balanced Accuracy,F1,상승 Recall,ROC AUC
0,Fold1,Expanding,0.647,0.604,0.712,0.707
1,Fold2,Expanding,0.612,0.405,0.270,0.679
2,Fold3,Expanding,0.473,0.151,0.093,0.524
3,Fold4,Expanding,0.415,0.098,0.073,0.307
4,Fold1,18개월 Rolling,0.678,0.632,0.729,0.742
5,Fold2,18개월 Rolling,0.572,0.304,0.190,0.720
6,Fold3,18개월 Rolling,0.460,0.214,0.147,0.567
7,Fold4,18개월 Rolling,0.482,0.354,0.364,0.422


# Classification V1 최종 확정

초기 실험에서는 향후 3개월 가격 변화율을 기준으로
하락 / 보합 / 상승의 3-Class Classification을 검토하였다.

최종 Classification V1은 시장 방향성 예측이라는 비즈니스 목적과
Rolling Backtest 결과를 기준으로 다음의 Binary Classification 문제로 확정한다.

- **0 = 비상승**
- **1 = 향후 3개월 가격 변화율 +3% 초과 상승**

## 최종 설정

- **Model**: XGBoost (`XGBClassifier`)
- **Target**: 향후 3개월 +3% 초과 상승 여부
- **Validation**: 시간순 Rolling Backtest
- **최종 Window**: 최근 18개월
- **평균 ROC-AUC**: 약 0.613
- **평균 Balanced Accuracy**: 약 0.548
- **평균 F1**: 약 0.376
- **평균 상승 Recall**: 약 0.357

Fold별 ROC-AUC는 약 `0.742 → 0.720 → 0.567 → 0.422`로,
시장 국면 변화에 따른 성능 편차가 크게 나타났다.

따라서 Classification V1은 **실서비스 최종 모델이 아니라 Experimental Baseline**으로 확정한다.

> 기존 실거래·전월세·금리·정책 및 과거 시장 흐름 Feature만으로도
> 일정 수준의 상승 신호는 확인했지만, 시장 국면 변화에 대한 안정성이 부족했다.

V2에서는 V1을 그대로 기준선으로 유지한 채
매매수급지수, 전세수급지수, 매매가격지수, 전세가격지수,
주택가격전망CSI, CCSI 등 선행 시장지표를 추가하고
동일한 Rolling Backtest로 개선 여부를 비교한다.

In [75]:
# ============================================================
# Classification V1 최종 재학습 + Artifact 저장
# ============================================================

import json
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier


V1_MODEL_DIR = (
    PROJECT_DIR
    / "models"
    / "classification"
    / "v1"
)

V1_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 1. 최종 Feature 확정
# ------------------------------------------------------------
FINAL_EXCLUDE_COLS = {
    "기준월",
    "시장방향",
    "향후3개월_변화율",
    BINARY_TARGET,
}

final_feature_cols = [
    col
    for col in classification_model_v1_df.columns
    if col not in FINAL_EXCLUDE_COLS
]

final_categorical_cols = ["구"]


# ------------------------------------------------------------
# 2. 최신 라벨 기준 최근 18개월 학습 데이터
# ------------------------------------------------------------
FINAL_WINDOW_MONTHS = 18

final_train_end = (
    classification_model_v1_df["기준월"]
    .max()
)

final_train_start = (
    final_train_end
    - (FINAL_WINDOW_MONTHS - 1)
)

final_train_df = (
    classification_model_v1_df[
        classification_model_v1_df["기준월"]
        .between(
            final_train_start,
            final_train_end,
        )
    ]
    .copy()
)


X_final_raw = final_train_df[
    final_feature_cols
]

y_final = final_train_df[
    BINARY_TARGET
]


# ------------------------------------------------------------
# 3. 전처리 + 최종 XGBoost 재학습
# ------------------------------------------------------------
final_preprocessor = ColumnTransformer(
    transformers=[
        (
            "gu",
            OneHotEncoder(
                handle_unknown="ignore",
            ),
            final_categorical_cols,
        ),
    ],
    remainder="passthrough",
)

X_final = (
    final_preprocessor
    .fit_transform(X_final_raw)
)


final_model = XGBClassifier(
    objective="binary:logistic",

    n_estimators=200,
    learning_rate=0.05,

    max_depth=3,
    min_child_weight=1,

    subsample=1.0,
    colsample_bytree=1.0,

    reg_alpha=0.0,
    reg_lambda=1.0,

    tree_method="hist",
    eval_metric="logloss",

    random_state=42,
    n_jobs=-1,
)


_ = final_model.fit(
    X_final,
    y_final,
    verbose=False,
)


# ------------------------------------------------------------
# 4. Backtest 기반 V1 Metrics
# ------------------------------------------------------------
v1_metrics = {
    "version": "v1",
    "status": "experimental_baseline",
    "model": "XGBClassifier",
    "target": "향후 3개월 가격 변화율 +3% 초과 상승 여부",
    "target_column": BINARY_TARGET,
    "validation": "18_month_rolling_backtest",
    "window_months": FINAL_WINDOW_MONTHS,

    "mean_roc_auc": float(
        rolling_18m_df["ROC AUC"].mean()
    ),
    "mean_balanced_accuracy": float(
        rolling_18m_df[
            "Balanced Accuracy"
        ].mean()
    ),
    "mean_f1": float(
        rolling_18m_df["F1"].mean()
    ),
    "mean_rise_recall": float(
        rolling_18m_df[
            "상승 Recall"
        ].mean()
    ),
    "min_roc_auc": float(
        rolling_18m_df["ROC AUC"].min()
    ),
    "max_roc_auc": float(
        rolling_18m_df["ROC AUC"].max()
    ),

    "final_train_start": str(
        final_train_start
    ),
    "final_train_end": str(
        final_train_end
    ),
    "final_train_rows": int(
        len(final_train_df)
    ),
    "feature_count": int(
        len(final_feature_cols)
    ),
}


# ------------------------------------------------------------
# 5. Artifact 저장
# ------------------------------------------------------------
joblib.dump(
    final_model,
    V1_MODEL_DIR
    / "xgb_classifier_v1.pkl",
)

joblib.dump(
    final_preprocessor,
    V1_MODEL_DIR
    / "preprocessor_v1.pkl",
)


with open(
    V1_MODEL_DIR
    / "feature_columns.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        final_feature_cols,
        file,
        ensure_ascii=False,
        indent=2,
    )


with open(
    V1_MODEL_DIR
    / "metrics.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        v1_metrics,
        file,
        ensure_ascii=False,
        indent=2,
    )


rolling_18m_df.to_csv(
    V1_MODEL_DIR
    / "rolling_results.csv",
    index=False,
    encoding="utf-8-sig",
)


rolling_backtest_df.to_csv(
    V1_MODEL_DIR
    / "rolling_window_all_results.csv",
    index=False,
    encoding="utf-8-sig",
)


window_summary.reset_index().to_csv(
    V1_MODEL_DIR
    / "rolling_window_summary.csv",
    index=False,
    encoding="utf-8-sig",
)


# ------------------------------------------------------------
# 6. 저장 결과 확인
# ------------------------------------------------------------
print("Classification V1 저장 완료")
print("저장 위치:", V1_MODEL_DIR)
print(
    "최종 학습 기간:",
    final_train_start,
    "~",
    final_train_end,
)
print(
    "최종 학습 데이터:",
    f"{len(final_train_df):,}",
)
print(
    "Feature:",
    len(final_feature_cols),
)
print(
    "평균 ROC-AUC:",
    f"{v1_metrics['mean_roc_auc']:.3f}",
)

print("\n저장 파일")
for path in sorted(
    V1_MODEL_DIR.iterdir()
):
    print("-", path.name)

Classification V1 저장 완료
저장 위치: c:\Users\uk\Apartment price\models\classification\v1
최종 학습 기간: 2024-10 ~ 2026-03
최종 학습 데이터: 450
Feature: 51
평균 ROC-AUC: 0.613

저장 파일
- feature_columns.json
- metrics.json
- preprocessor_v1.pkl
- rolling_results.csv
- rolling_window_all_results.csv
- rolling_window_summary.csv
- xgb_classifier_v1.pkl
